# M1 v2 — Additive Multi-Loss Grid Search

## Rationale (Phase 1 findings)
- **E1 (WSFT)**: Most balanced — best DA after E3, strong completeness, good format
- **E7 (Decision-Only)**: Best composite-5, best clinical correctness + reasoning
- **E6 (Explanation-Only)**: Best reasoning_mean, best coherence, highest CC
- **E5b (Explanation Entropy)**: Best balanced entropy method, strong safety + CC

## Combinations tested

| Combo | Losses | Why |
|-------|--------|-----|
| A | L_wsft + L_dec_only | E1+E7: balanced coverage + deep reasoning. Two losses = less interference |
| B | L_wsft + L_dec_only + L_expl_ent | E1+E7+E5b: adds KD signal (teacher logprobs) to Combo A |
| C | L_dec_only + L_expl_only | E7+E6: section specialists — decision + explanation cover full answer |

Each combo tested with 3 weight configurations → 9 runs per model size.

## Training: 2 epochs (same as Phase 1)

In [1]:
import shared_utils, inspect
print("shared_utils path:", shared_utils.__file__)
print(inspect.getsource(shared_utils.load_student))

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shared_utils path: c:\Users\adishalit1\Desktop\kd_project\shared_utils.py
def load_student(model_name: str, cfg: dict):
    """Load tokenizer + LoRA model. Uses 8-bit for 7B, bf16 for smaller."""
    print(f"  Loading {model_name} from {cfg['path']}...")
    tokenizer = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if "7b" in model_name:
        print("  Using 8-bit quantization (Windows stability)")
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            cfg["path"],
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            cfg["path"],
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )

    lora_cfg = Lor

In [2]:
# Cell 0: Environment + Model Selection (Linux/WSL2)
import os, torch

TRAIN_MODEL = "qwen25_1p5b_base"
#TRAIN_MODEL = "qwen25_3b_base"
#TRAIN_MODEL = "qwen25_7b_base"

use_bf16 = "7b" not in TRAIN_MODEL

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"Will train: {TRAIN_MODEL} | bf16={use_bf16}")

Will train: qwen25_3b_base | bf16=True
PyTorch: 2.6.0+cu124
CUDA build: 12.4
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
Restart kernel before switching to a different model or combo


In [3]:
# Cell 0.5: CUDA smoke test
import torch, gc

assert torch.cuda.is_available(), "CUDA is not available"

x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
print("Smoke test OK | norm =", y.norm().item())

del x, y
gc.collect()
torch.cuda.empty_cache()

Smoke test OK | norm = 262144.0


In [4]:
# Cell 1: Imports + data
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, EPOCHS, LR, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA,
    STUDENTS, load_data, load_student,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, find_expl_span_chars,
    teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m1v2_additive_grid")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Grid of combinations ──
COMBOS = {
    "A1": {"wsft": 0.7, "dec": 0.3},
    "A2": {"wsft": 0.5, "dec": 0.5},
    "A3": {"wsft": 0.3, "dec": 0.7},
    "B1": {"wsft": 0.5, "dec": 0.3, "expl_ent": 0.2},
    "B2": {"wsft": 0.4, "dec": 0.4, "expl_ent": 0.2},
    "B3": {"wsft": 0.3, "dec": 0.5, "expl_ent": 0.2},
    "C1": {"dec": 0.6, "expl_only": 0.4},
    "C2": {"dec": 0.5, "expl_only": 0.5},
    "C3": {"dec": 0.4, "expl_only": 0.6},
}

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)
print(f"Mean confidence: {MEAN_C:.6f}")
print(f"Combos to run: {list(COMBOS.keys())}")

✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
Mean confidence: 0.999763
Combos to run: ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3']


In [5]:
# Cell 2: Dataset — precomputes all fields needed by any combo
class M1v2DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        print("  Precomputing dataset (one-time cost)...")
        self.items = []
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]

            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            input_ids, offsets, labels, answer_start = tokenize_and_mask(
                tokenizer, prompt_text, answer
            )

            # ── WSFT weights (for L_wsft) ──
            wsft_weights = [0.0] * len(input_ids)
            spans = get_section_spans(answer)
            dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
            expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]

            for i, (s, e) in enumerate(offsets):
                if e <= s: continue
                if s >= answer_start:
                    w = W_FORMAT
                    if in_any_span(s, e, dec_spans): w = W_DECISION
                    if in_any_span(s, e, expl_spans): w = W_EXPL
                    wsft_weights[i] = float(w)

            active_w = [w for w in wsft_weights if w > 0]
            if active_w:
                mw = sum(active_w) / len(active_w)
                if mw > 1e-6:
                    wsft_weights = [w / mw if w > 0 else 0.0 for w in wsft_weights]

            # ── Decision mask (for L_dec_only) ──
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span_chars = find_decision_span_chars(answer)
            if dec_span_chars:
                ds, de = dec_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + ds or s >= answer_start + de):
                        dec_mask[i] = True

            # ── Explanation mask (for L_expl_only) ──
            expl_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            expl_span_chars = find_expl_span_chars(answer)
            if expl_span_chars:
                es, ee = expl_span_chars
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start + es or s >= answer_start + ee):
                        expl_mask[i] = True

            # ── Teacher entropy on explanation section (for L_expl_ent) ──
            ent_teacher_expl = teacher_section_entropy_mean(r, expl_span_chars)

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
                "dec_mask": dec_mask,
                "expl_mask": expl_mask,
                "ent_teacher_expl": ent_teacher_expl.float(),
            })
        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Dataset class ready")

✅ Dataset class ready


In [6]:
# # Cell 3: Trainer — computes whichever losses the current combo needs
# class M1v2Trainer(Trainer):
#     def __init__(self, combo_weights, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.combo_weights = combo_weights
#         self._ce = torch.nn.CrossEntropyLoss(reduction="none")  # ignore_index stays -100 by default

#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         labels = inputs.pop("labels")
#         wsft_w = inputs.pop("wsft_weights")
#         dec_mask = inputs.pop("dec_mask")
#         expl_mask = inputs.pop("expl_mask")
#         ent_teacher_expl = inputs.pop("ent_teacher_expl")

#         outputs = model(**inputs)
#         logits = outputs.logits

#         sl = logits[:, :-1, :].contiguous()
#         ll = labels[:, 1:].contiguous()
#         sw = wsft_w[:, 1:].contiguous()
#         dm = dec_mask[:, 1:]
#         em = expl_mask[:, 1:]

#         B, S, V = sl.size()

#         tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(B, S)
#         active = (ll != -100).float()

#         loss = torch.tensor(0.0, device=sl.device)
#         cw = self.combo_weights

#         # L_wsft: section-weighted SFT
#         if "wsft" in cw:
#             w = sw * active
#             denom = active.sum(dim=1).clamp(min=1.0)
#             L_wsft = (tok_loss * w).sum(dim=1) / denom
#             loss = loss + cw["wsft"] * L_wsft.mean()

#         # L_dec_only: decision-token supervision
#         if "dec" in cw:
#             da = dm.float() * active
#             L_dec = (tok_loss * da).sum() / da.sum().clamp(min=1.0)
#             loss = loss + cw["dec"] * L_dec

#         # L_expl_only: explanation-token supervision
#         if "expl_only" in cw:
#             ea = em.float() * active
#             L_expl = (tok_loss * ea).sum() / ea.sum().clamp(min=1.0)
#             loss = loss + cw["expl_only"] * L_expl

#         # L_expl_ent: explanation entropy matching
#         if "expl_ent" in cw:
#             expl_active = em & (ll != -100)
#             ent_student = torch.zeros(B, device=sl.device)
#             for b in range(B):
#                 idx = expl_active[b].nonzero(as_tuple=True)[0]
#                 if len(idx) > 0:
#                     p = torch.softmax(sl[b, idx, :], dim=-1)
#                     ent_student[b] = -(p * torch.log(p + 1e-9)).sum(-1).mean()
#             L_ent = LAMBDA * ((ent_student - ent_teacher_expl.to(sl.device)) ** 2).mean()
#             loss = loss + cw["expl_ent"] * L_ent

#         return (loss, outputs) if return_outputs else loss

# print("✅ Trainer ready")

In [7]:
# Cell 3: Trainer — faster + safer entropy branch
import torch
import torch.nn.functional as F

ENT_MAX_TOKENS = 32   # try 16 if it still crashes; 32 is a good first test

class M1v2Trainer(Trainer):
    def __init__(self, combo_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.combo_weights = combo_weights
        self._ce = torch.nn.CrossEntropyLoss(reduction="none")  # ignore_index=-100 by default

    def _subsample_idx(self, idx, max_tokens=ENT_MAX_TOKENS):
        if idx.numel() <= max_tokens:
            return idx
        # deterministic even spacing, keeps runs reproducible
        pos = torch.linspace(0, idx.numel() - 1, steps=max_tokens, device=idx.device)
        return idx[pos.long()]

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_w = inputs.pop("wsft_weights")
        dec_mask = inputs.pop("dec_mask")
        expl_mask = inputs.pop("expl_mask")
        ent_teacher_expl = inputs.pop("ent_teacher_expl")

        outputs = model(**inputs)
        logits = outputs.logits

        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        sw = wsft_w[:, 1:].contiguous()
        dm = dec_mask[:, 1:]
        em = expl_mask[:, 1:]

        B, S, V = sl.size()

        tok_loss = self._ce(sl.view(-1, V), ll.view(-1)).view(B, S)
        active = (ll != -100).float()

        loss = torch.tensor(0.0, device=sl.device)
        cw = self.combo_weights

        # L_wsft
        if "wsft" in cw:
            w = sw * active
            denom = active.sum(dim=1).clamp(min=1.0)
            L_wsft = (tok_loss * w).sum(dim=1) / denom
            loss = loss + cw["wsft"] * L_wsft.mean()

        # L_dec_only
        if "dec" in cw:
            da = dm.float() * active
            L_dec = (tok_loss * da).sum() / da.sum().clamp(min=1.0)
            loss = loss + cw["dec"] * L_dec

        # L_expl_only
        if "expl_only" in cw:
            ea = em.float() * active
            L_expl = (tok_loss * ea).sum() / ea.sum().clamp(min=1.0)
            loss = loss + cw["expl_only"] * L_expl

        # L_expl_ent
        # IMPORTANT: use only a sampled subset of explanation positions
        if "expl_ent" in cw:
            expl_active = em & (ll != -100)
            ent_vals = []

            for b in range(B):
                idx = expl_active[b].nonzero(as_tuple=True)[0]

                if idx.numel() == 0:
                    ent_vals.append(torch.zeros((), device=sl.device, dtype=torch.float32))
                    continue

                idx = self._subsample_idx(idx)

                # Compute entropy in fp32 for stability
                z = sl[b, idx, :].float()          # [T, V]
                logp = torch.log_softmax(z, dim=-1)
                p = logp.exp()
                ent_b = -(p * logp).sum(dim=-1).mean()   # scalar
                ent_vals.append(ent_b)

            ent_student = torch.stack(ent_vals)  # [B]
            target = ent_teacher_expl.to(sl.device, dtype=ent_student.dtype)
            L_ent = LAMBDA * F.mse_loss(ent_student, target)
            loss = loss + cw["expl_ent"] * L_ent

        return (loss, outputs) if return_outputs else loss

print("✅ Trainer ready")
print(f"Entropy branch will use at most {ENT_MAX_TOKENS} explanation tokens per sample")

✅ Trainer ready
Entropy branch will use at most 32 explanation tokens per sample


In [8]:
# Cell 3.5: CUDA memory helpers
import gc
import torch

def gpu_mem(tag=""):
    if not torch.cuda.is_available():
        print(f"[{tag}] CUDA not available")
        return
    alloc = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_alloc = torch.cuda.max_memory_allocated() / 1024**3
    max_reserved = torch.cuda.max_memory_reserved() / 1024**3
    print(
        f"[{tag}] "
        f"alloc={alloc:.2f} GB | reserved={reserved:.2f} GB | "
        f"max_alloc={max_alloc:.2f} GB | max_reserved={max_reserved:.2f} GB"
    )

def hard_cleanup(trainer=None, model=None, tokenizer=None):
    try:
        if trainer is not None and hasattr(trainer, "accelerator"):
            trainer.accelerator.free_memory()
    except Exception as e:
        print(f"accelerator.free_memory() warning: {e}")

    try:
        if trainer is not None:
            trainer.model = None
            trainer.model_wrapped = None
            trainer.optimizer = None
            trainer.lr_scheduler = None
            trainer.train_dataset = None
            trainer.data_collator = None
    except Exception as e:
        print(f"trainer cleanup warning: {e}")

    try:
        del trainer, model, tokenizer
    except Exception:
        pass

    import gc
    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except RuntimeError as e:
            print(f"empty_cache skipped after CUDA error: {e}")

        try:
            torch.cuda.reset_peak_memory_stats()
        except RuntimeError as e:
            print(f"reset_peak_memory_stats skipped after CUDA error: {e}")

        try:
            gpu_mem("after_cleanup")
        except Exception:
            pass

# def hard_cleanup(trainer=None, model=None, tokenizer=None):
#     try:
#         if trainer is not None and hasattr(trainer, "accelerator"):
#             # Accelerate docs recommend free_memory/clear between trainings
#             trainer.accelerator.free_memory()
#     except Exception as e:
#         print(f"accelerator.free_memory() warning: {e}")

#     try:
#         if trainer is not None:
#             trainer.model = None
#             trainer.model_wrapped = None
#             trainer.optimizer = None
#             trainer.lr_scheduler = None
#             trainer.train_dataset = None
#             trainer.data_collator = None
#     except Exception as e:
#         print(f"trainer cleanup warning: {e}")

#     del trainer, model, tokenizer
#     gc.collect()

#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.reset_peak_memory_stats()
#         gpu_mem("after_cleanup")

In [9]:
def find_training_collisions(root):
    hits = []
    for name, module in root.named_modules():
        child_keys = list(getattr(module, "_modules", {}).keys())
        if "training" in child_keys:
            hits.append(name if name else "<root>")
    return hits

In [ ]:
# Cell 4: Train remaining combos
import os, gc, torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments

cfg = STUDENTS[TRAIN_MODEL]

def gpu_mem(tag):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_alloc = torch.cuda.max_memory_allocated() / 1024**3
        max_reserved = torch.cuda.max_memory_reserved() / 1024**3
        print(f"[{tag}] alloc={alloc:.2f} GB | reserved={reserved:.2f} GB | max_alloc={max_alloc:.2f} GB | max_reserved={max_reserved:.2f} GB")

# Check which combos already done
done = set()
for combo_name in COMBOS:
    marker = os.path.join(OUT_DIR, TRAIN_MODEL, combo_name, "adapter_config.json")
    if os.path.exists(marker):
        done.add(combo_name)
        print(f"⏩ {combo_name} already done")

remaining = [c for c in COMBOS if c not in done]
print(f"\nWill run {len(remaining)} combos: {remaining}")

# For debugging, you can force a single combo:
# remaining = ["B1"]

# Precompute dataset once
tokenizer_for_ds = AutoTokenizer.from_pretrained(cfg["path"], trust_remote_code=True)
if tokenizer_for_ds.pad_token is None:
    tokenizer_for_ds.pad_token = tokenizer_for_ds.eos_token

dataset = M1v2DatasetFast(teacher_rows, prompts, tokenizer_for_ds, cfg["is_instruct"], MEAN_C)
collator = FlexibleCollator(
    tokenizer_for_ds,
    extra_1d_fields=["wsft_weights", "dec_mask", "expl_mask"],
    extra_scalar_fields=["ent_teacher_expl"],
)

gpu_mem("after_dataset")

for combo_name in remaining:
    cw = COMBOS[combo_name]
    print(f"\n{'='*60} M1v2: {TRAIN_MODEL} / {combo_name} = {cw} {'='*60}")

    out_path = os.path.join(OUT_DIR, TRAIN_MODEL, combo_name)
    os.makedirs(out_path, exist_ok=True)

    gpu_mem(f"before_load_{combo_name}")
    tokenizer, model = load_student(TRAIN_MODEL, cfg)

    hits = find_training_collisions(model)
    print("training-key collisions:", len(hits))
    for h in hits[:20]:
        print(" -", h)

    assert len(hits) == 0, "Found a child module named 'training' inside the model"

    if getattr(model.config, "use_cache", None) is not None:
        model.config.use_cache = False

    gpu_mem(f"after_load_{combo_name}")

    trainer = M1v2Trainer(
        combo_weights=cw,
        model=model,
        args=TrainingArguments(
            output_dir=out_path,
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=8,
            gradient_accumulation_steps=4,
            learning_rate=LR,
            logging_steps=50,
            save_strategy="no",
            bf16=use_bf16,
            tf32=True,
            seed=SEED,
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=4,
            gradient_checkpointing=("7b" in TRAIN_MODEL),
        ),
        train_dataset=dataset,
        data_collator=collator,
    )

    gpu_mem(f"before_train_{combo_name}")
    trainer.train()
    gpu_mem(f"after_train_{combo_name}")

    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"✅ Saved {combo_name} → {out_path}")

    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gpu_mem(f"after_cleanup_{combo_name}")

print(f"\n✅ M1v2 complete for {TRAIN_MODEL}.")

⏩ A1 already done
⏩ A2 already done
⏩ A3 already done

Will run 6 combos: ['B1', 'B2', 'B3', 'C1', 'C2', 'C3']
  Precomputing dataset (one-time cost)...


  Tokenizing: 100%|██████████| 5000/5000 [00:23<00:00, 213.18it/s]


  ✅ Precomputed 5000 samples
[after_dataset] alloc=0.01 GB | reserved=0.02 GB | max_alloc=0.07 GB | max_reserved=0.08 GB

============================================================ M1v2: qwen25_3b_base / B1 = {'wsft': 0.5, 'dec': 0.3, 'expl_ent': 0.2} ============================================================
[before_load_B1] alloc=0.01 GB | reserved=0.02 GB | max_alloc=0.07 GB | max_reserved=0.08 GB
  Loading qwen25_3b_base from Qwen/Qwen2.5-3B...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:04<00:00, 101.62it/s]


  ✅ qwen25_3b_base: 7,372,800 trainable / 3,093,311,488 total params
training-key collisions: 0
[after_load_B1] alloc=5.78 GB | reserved=5.86 GB | max_alloc=5.78 GB | max_reserved=5.86 GB
[before_train_B1] alloc=5.78 GB | reserved=5.86 GB | max_alloc=5.78 GB | max_reserved=5.86 GB


Step,Training Loss
